# 09 Q2 descriptive runs (preregistration 9.8)

Run as **Save Version → Save & Run All**. About 3-3.5 hours. Every cell runs once, top to bottom; nothing to edit.

When it is done: saved version → **Output** tab → download `q2_descriptive.zip`.

Runs (`helixadd.q2.run_descriptive`, saved to `results/q2_descriptive/`):
- clamp test: digit swap with the last-digit supernode held at its clean values;
- each of the 7 periods swapped alone, and the two groups on the same prompts;
- digit swap that flips the carry (addition);
- sub-helix patching into a' + b (Kantamneni-style);
- one point at a time, with 20 random sets per point, on 200 prompts.
The comparison supernode is watched too (addition).

Setup: GPU T4, Internet on, secret `HF_TOKEN` ticked.

## 1. Get the code

In [ ]:
!rm -rf helix_addition && git clone https://github.com/amark-23/helix_addition.git
!pip install -q git+https://github.com/safety-research/circuit-tracer@v0.5.3

## 2. Work inside the repo, log in

In [ ]:
import os, sys
os.chdir("/kaggle/working/helix_addition")
sys.path.insert(0, "/kaggle/working/helix_addition/src")
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(UserSecretsClient().get_secret("HF_TOKEN"))

## 3. Model, number vectors, prompts

In [ ]:
import numpy as np, time
from helixadd import model as hmodel, q1, q2
model = hmodel.load_model()
data = dict(np.load("results/number_acts/data.npz"))
pairs = q1.eval_pairs()
print(len(pairs), "evaluation prompts")

## 4. Check the clamp: with nothing swapped, clamping last digit to its clean values must change nothing

In [ ]:
watch = q2.supernode_groups()
quads = q2.swap_pairs("digit", "a", pairs[:32])
ref = q2.reference(model, quads, watch)
dirs = q2.point_directions(data, "a")
none = [np.zeros((0, 2048))] * len(q2.POINTS)
df = q2.run_one(model, quads, "a", dirs, none, ref, watch, clamp=(watch["last digit"], ref["clean"]["last digit"]))
print("answers unchanged:", (df["outcome"] == "N").mean(), " largest magnitude change:", round(float(df["change magnitude"].max()), 4))

## 5. All descriptive runs

In [ ]:
t = time.time()
q2.run_descriptive(model, data, pairs)
print(f"{(time.time() - t) / 3600:.1f} hours")

## 6. Pack the results for download

In [ ]:
!cd /kaggle/working && zip -qr q2_descriptive.zip helix_addition/results/q2_descriptive && ls -lh q2_descriptive.zip